<a href="https://colab.research.google.com/github/YOUR-USERNAME/bags-vectors-transformers/blob/main/day2/notebooks/2_bow_limits_exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bags, Vectors & Transformers
## Day 2 — Seeing the Limits of Bag-of-Words

**A Methods Workshop in Computational Text Analysis**
Denise J. Roth · Strategic Communication Group · Wageningen University & Research

---

We have spent a lot of time building and using bag-of-words representations. They are
genuinely useful — but they rest on a strong simplification: **a document is just an
unordered pile of word counts**.

In this notebook we make the consequences of that simplification **concrete**. Rather than
take the drawbacks on faith, we will *demonstrate* each one in code and watch bag-of-words
fail in ways you can see for yourself.

We will look at five limitations:

1. **Word order is lost**
2. **No sense of meaning** (synonyms look unrelated)
3. **High dimensionality and sparsity**
4. **Out-of-vocabulary words**
5. **One meaning per word** (context is ignored)

Everything here runs on small hardcoded examples — no external data needed.

> Run each cell in order with `Shift + Enter`. Each section ends with a short **✏️ Exercise**.


## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("Setup complete!")

### A tiny helper

We will repeatedly turn a few sentences into a document-term matrix and look at it as a
table. This helper does that so we can focus on the *ideas*, not the plumbing.


In [ ]:
def make_dtm(docs, **kwargs):
    """Return a DTM as a readable DataFrame (rows = docs, columns = terms)."""
    vec = CountVectorizer(**kwargs)
    X = vec.fit_transform(docs)
    return pd.DataFrame(
        X.toarray(),
        columns=vec.get_feature_names_out(),
        index=[f"D{i+1}" for i in range(len(docs))],
    )

# Quick check
make_dtm(["the cat sat", "the dog sat"])

## 1. Word order is lost

The claim: bag-of-words sees only *counts*, not *sequence*. Two sentences with the same
words in a different order become **identical**. Let's prove it.


In [ ]:
pair = [
    "the dog bit the man",
    "the man bit the dog",
]

dtm = make_dtm(pair)
dtm

Look at the two rows. They are **exactly the same** — every count is identical. Yet the
two sentences describe opposite events. Let's confirm the vectors are identical by measuring
their similarity.


In [ ]:
vec = CountVectorizer()
X = vec.fit_transform(pair)

sim = cosine_similarity(X)[0, 1]
print(f"Cosine similarity between the two sentences: {sim:.3f}")
print("(1.000 means the model considers them identical.)")

A similarity of **1.0** — the model literally cannot tell "the dog bit the man" from
"the man bit the dog". All the meaning carried by word *order* (who did what to whom) is gone.


> **✏️ Exercise 1**
>
> Come up with your own pair of sentences that mean **different things** but use the **same
> words**. Put them in a list and check their cosine similarity. Can you get anything other
> than 1.0? *(You can't, as long as the words and counts match — that's the point.)*


In [ ]:
# Your code here


## 2. No sense of meaning (synonyms look unrelated)

The claim: to bag-of-words, every word is a separate symbol. Words that mean the *same
thing* are treated as completely unrelated. Let's show it with three sentences.


In [ ]:
sentences = [
    "I am very happy",       # D1
    "I am very joyful",      # D2  (synonym of happy)
    "I am very sad",         # D3  (opposite of happy)
]

dtm = make_dtm(sentences)
dtm

Now here is the key question: **which two sentences are most similar in meaning?**
Obviously D1 ("happy") and D2 ("joyful"). Let's see what bag-of-words thinks.


In [ ]:
vec = CountVectorizer()
X = vec.fit_transform(sentences)

sims = cosine_similarity(X)
sim_df = pd.DataFrame(sims, index=["D1 happy", "D2 joyful", "D3 sad"],
                      columns=["D1 happy", "D2 joyful", "D3 sad"])
print("Cosine similarity between sentences:")
sim_df.round(3)

Look at the off-diagonal numbers. The similarity between **"happy" and "joyful"** is
**exactly the same** as between **"happy" and "sad"** — even though one pair is synonyms and
the other is opposites!

To bag-of-words, `happy`, `joyful`, and `sad` are just three different columns. It has no
idea that two of them mean nearly the same thing. The shared words ("I am very") carry all
the similarity; the actual sentiment word contributes nothing to relatedness.


In [ ]:
print(f"happy  vs joyful: {sims[0,1]:.3f}")
print(f"happy  vs sad:    {sims[0,2]:.3f}")
print("\nSame number — the model cannot tell synonyms from opposites.")

> **✏️ Exercise 2**
>
> Add a fourth sentence `"I am very cheerful"` (another synonym of happy). Recompute the
> similarity matrix. Is "cheerful" any closer to "happy" than "sad" is? Explain in a comment
> why not.


In [ ]:
# Your code here


## 3. High dimensionality and sparsity

The claim: real DTMs have one column per vocabulary word — tens of thousands — and any one
document uses only a tiny fraction, so the matrix is *mostly zeros*. Let's watch the vocabulary
(and the emptiness) grow as we add documents.


In [ ]:
# A slightly bigger toy corpus: each doc is short, but they use different words
corpus = [
    "the economy grew last quarter",
    "climate policy dominated the debate",
    "the football match ended in a draw",
    "researchers published a new study",
    "the election results surprised everyone",
    "inflation affected household budgets",
    "the orchestra performed a symphony",
    "students protested tuition fees",
]

dtm = make_dtm(corpus)
print("DTM shape (documents x terms):", dtm.shape)
dtm

Even with just 8 short sentences, notice how **wide** the table is and how many **zeros**
it contains. Each sentence lights up only a handful of columns. Let's quantify the sparsity.


In [ ]:
total_cells = dtm.size
zero_cells = (dtm == 0).sum().sum()
print(f"Total cells:      {total_cells}")
print(f"Zero cells:       {zero_cells}")
print(f"Percentage zeros: {100 * zero_cells / total_cells:.1f}%")

Now imagine this at realistic scale. Let's simulate how the vocabulary grows as a corpus
gets bigger — this is why real DTMs explode in width.


In [ ]:
# Simulate vocabulary growth using our corpus repeated with slight variation
np.random.seed(0)
sizes = [2, 4, 6, 8]
vocab_sizes = []
sparsities = []

for n in sizes:
    d = make_dtm(corpus[:n])
    vocab_sizes.append(d.shape[1])
    sparsities.append(100 * (d == 0).sum().sum() / d.size)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(sizes, vocab_sizes, marker="o", color="#34B233")
ax1.set_title("Vocabulary grows with corpus size")
ax1.set_xlabel("Number of documents")
ax1.set_ylabel("Vocabulary size (columns)")
ax1.grid(alpha=0.3)

ax2.plot(sizes, sparsities, marker="o", color="#1A1A2E")
ax2.set_title("And the matrix gets emptier")
ax2.set_xlabel("Number of documents")
ax2.set_ylabel("Percentage of zero cells")
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

The vocabulary keeps growing as you add documents (each new document tends to introduce
new words), but any single document still uses only a few words — so the **proportion of
zeros climbs**. At the scale of real corpora (tens of thousands of words), the DTM is well
over 99% zeros. That is expensive to store and statistically awkward to work with.


> **✏️ Exercise 3**
>
> Add two or three of your own sentences to `corpus`, each on a *new* topic with words not
> already used. Rebuild the DTM. Does the vocabulary get wider and the sparsity higher?


In [ ]:
# Your code here


## 4. Out-of-vocabulary words

The claim: a bag-of-words model only knows words it saw when the vectorizer was *fitted*.
A new word at prediction time is simply **invisible** — dropped as if it were never there.
Let's demonstrate.


In [ ]:
# Fit the vectorizer on a training sentence
train_docs = ["the policy is good"]
vec = CountVectorizer()
vec.fit(train_docs)

print("Vocabulary the model knows:", list(vec.get_feature_names_out()))

In [ ]:
# Now transform a NEW sentence containing words the model never saw
new_doc = ["the policy is catastrophic"]   # "catastrophic" is new
X_new = vec.transform(new_doc)

result = pd.DataFrame(X_new.toarray(), columns=vec.get_feature_names_out(), index=["new_doc"])
print("New sentence:", new_doc[0])
print()
print("How the model represents it:")
print(result)

Look carefully. The new sentence was *"the policy is catastrophic"* — a strongly negative
statement. But **"catastrophic" does not appear as a column**, because the model never saw it
during fitting. It is silently **dropped**. The model's representation of the sentence is
identical to what it would be for *"the policy is"* — the most important word vanished.


In [ ]:
# Prove it: the strongly-negative new sentence looks identical to a bland one
bland = vec.transform(["the policy is"])
catastrophic = vec.transform(["the policy is catastrophic"])

print("Are the two vectors identical?",
      np.array_equal(bland.toarray(), catastrophic.toarray()))
print("\n'catastrophic' contributed nothing — it was out of vocabulary.")

> **✏️ Exercise 4**
>
> Transform the sentence `"the wonderful brilliant policy"` using the same `vec` (fitted only
> on "the policy is good"). How many of its words survive? What does that tell you about
> applying a fitted model to new, real-world text full of unseen words?


In [ ]:
# Your code here


## 5. One meaning per word (context is ignored)

The claim: bag-of-words gives each word a single column, regardless of meaning. A word with
two meanings (like **"bank"** — river vs. money) is collapsed into one, and the model cannot
tell the senses apart. Let's show it.


In [ ]:
sentences = [
    "I sat on the river bank",       # D1: bank = riverside
    "I fished from the river bank",  # D2: bank = riverside
    "I deposited cash at the bank",  # D3: bank = financial
]

dtm = make_dtm(sentences)
dtm

There is a single **"bank"** column, and all three sentences put a 1 in it — even though
D1/D2 mean *riverside* and D3 means *financial institution*. The model treats all three uses
as the same thing.

Worse, watch what drives similarity. Intuitively D1 and D2 (both riverside, both fishing/
sitting by water) are about the same scene, and D3 is different. But the model only sees
shared *word strings*.


In [ ]:
vec = CountVectorizer()
X = vec.fit_transform(sentences)
sims = cosine_similarity(X)

sim_df = pd.DataFrame(
    sims.round(3),
    index=["D1 river", "D2 river", "D3 money"],
    columns=["D1 river", "D2 river", "D3 money"],
)
print("Similarity — note it is driven purely by shared word strings:")
sim_df

The model links the sentences only through the literal token **"bank"** (and "the", "i").
It has **no representation** of the fact that D1 and D2 share a *meaning* of "bank" that D3
does not. Both senses collapse into one column, so context — the very thing that
disambiguates them — is thrown away.


> **✏️ Exercise 5**
>
> Write three sentences using the word **"spring"** in different senses (the season, a coil,
> and to jump). Build the DTM. Confirm there is only **one** "spring" column, and explain in
> a comment why that is a problem for meaning.


In [ ]:
# Your code here


## Wrap-up

You have now *seen*, not just heard, the core limitations of bag-of-words:

1. **Word order is lost** — "dog bites man" = "man bites dog" (similarity 1.0)
2. **No sense of meaning** — "happy" is no closer to "joyful" than to "sad"
3. **High dimensionality and sparsity** — the matrix balloons and fills with zeros
4. **Out-of-vocabulary words** — unseen words silently vanish
5. **One meaning per word** — "bank" (river) and "bank" (money) share one column

Every one of these traces back to the same root: bag-of-words treats **words as isolated
symbols** with no relationship to one another and no sensitivity to order or context.

That is not a reason to abandon bag-of-words — it remains a transparent, fast, and often
perfectly adequate tool. But knowing exactly *where* it breaks is what lets you choose the
right method for your question, and it is the motivation behind the techniques that follow.

### Optional challenge

Take the "dog bites man" example and try to *rescue* word order using **bigrams**
(pairs of consecutive words). Rebuild the DTM with `make_dtm(pair, ngram_range=(1, 2))` and
recompute the cosine similarity. Does it drop below 1.0? By how much? Then think: what does
this fix cost you? *(Hint: look at how many columns the bigram DTM has.)*


In [ ]:
# Optional challenge — your code here
